In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :powerlaw

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 1

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [powerlaw_model] Fitting chain 1 (tau=59)
[ Info: [powerlaw] iter 1000/1000000 elapsed=5.6s, rate=0.111, mean=[0.680, 0.00212, 0.427, 0.379], std=[0.1426, 0.000381, 0.0277, 0.0459] [ADAPT]
[ Info: [powerlaw] iter 2000/1000000 elapsed=10.3s, rate=0.103, mean=[0.631, 0.00192, 0.415, 0.446], std=[0.1105, 0.000352, 0.0231, 0.0723] [ADAPT]
[ Info: [powerlaw] iter 3000/1000000 elapsed=14.3s, rate=0.093, mean=[0.636, 0.00166, 0.411, 0.508], std=[0.0919, 0.000436, 0.0200, 0.0991] [ADAPT]
[ Info: [powerlaw] iter 4000/1000000 elapsed=18.3s, rate=0.084, mean=[0.640, 0.00151, 0.406, 0.544], std=[0.0801, 0.000448, 0.0196, 0.1024] [ADAPT]
[ Info: [powerlaw] iter 5000/1000000 elapsed=22.2s, rate=0.078, mean=[0.644, 0.00139, 0.398, 0.571], std=[0.0723, 0.000455, 0.0232, 0.1046] [ADAPT]
[ Info: [powerlaw] iter 6000/1000000 elapsed=26.2s, rate=0.073, mean=[0.650, 0.00131, 0.396, 0.591], std=[0.0671, 0.000447, 0.0218, 0.1032] [ADAPT]
[ Info: [powerlaw] iter 7000/1000000 elapsed=30.2s, rate=0.072,